# AMD-stability criterion — example

Companion notebook for **amdrepertory**.

We compute, for a three-planet system:
- the optimal eccentricity-like vector $c = (c_1, c_2, c_3)$,
- the **actual** AMD,
- the **critical** AMD,
- the ratio $\beta = \mathrm{AMD}/\mathrm{AMD_{crit}}$  ($\beta<1$ stable, $\beta>1$ unstable).

Two paths are shown: (A) directly from periods/masses, and (B) from a `rebound` simulation.


In [2]:
import numpy as np
import amdrepertory as nar
from amdrepertory import optimize_amd_crit, amd_crit_from_sim, AMDOptimizationError

ModuleNotFoundError: No module named 'amdrepertory'

## A. From periods and masses

Normalized units: central mass `m0=1`, `G=1`. Planet masses are then in units of the central mass.

In [2]:
periods = [1.0, 1.3, 1.7]      # P1, P2, P3
masses  = [1e-5, 1e-5, 1e-5]   # m1, m2, m3 (in units of m0)

out = optimize_amd_crit(periods, masses, verbose=True)

print()
print("c        =", out["c"])
print("AMD_crit =", out["AMD_crit"])
print("|tau-1|  =", out["tau_residual"])
print("ICs in agreement:", out["n_ic_agree"])

IC#0: success=True c=[0.02328637 0.01293774 0.02444064] |tau-1|=9.07e-12 feasible=True msg='`gtol` termination condition is satisfied.'
IC#1: success=True c=[0.02368572 0.01245542 0.02501051] |tau-1|=4.67e-13 feasible=True msg='`gtol` termination condition is satisfied.'

c        = [0.02368572 0.01245542 0.02501051]
AMD_crit = 4.001807835071924e-09
|tau-1|  = 4.669598041573408e-13
ICs in agreement: 2


### Computing beta from an actual AMD

`optimize_amd_crit` gives `AMD_crit`. To get $\beta$ you need the system's
**actual** AMD. Below we compute it analytically from the eccentricities
(here a toy example with small eccentricities).

In [3]:
def amd_from_elements(periods, masses, eccs, m0=1.0, G=1.0):
    """Actual AMD = sum_i Lambda_i (1 - sqrt(1-e_i^2)), no inclination."""
    mu = G * m0
    amd = 0.0
    for P, m, e in zip(periods, masses, eccs):
        a = (mu * P**2 / (4*np.pi**2))**(1/3)
        Lam = m * (mu * a)**0.5
        amd += Lam * (1 - (1 - e**2)**0.5)
    return amd

eccs = [0.01, 0.005, 0.01]
AMD_actual = amd_from_elements(periods, masses, eccs)

beta = AMD_actual / out["AMD_crit"]
print(f"AMD (actual) = {AMD_actual:.4e}")
print(f"AMD_crit     = {out['AMD_crit']:.4e}")
print(f"beta         = {beta:.4f}  ->", "STABLE" if beta < 1 else "UNSTABLE")

AMD (actual) = 6.6830e-10
AMD_crit     = 4.0018e-09
beta         = 0.1670  -> STABLE


## B. From a rebound simulation

`amd_crit_from_sim` reads masses and periods from the simulation, runs the
optimization, and returns everything including `beta` and `stable`.

> Requires `rebound`: `pip install rebound`

In [4]:
try:
    import rebound as rb

    sim = rb.Simulation()
    sim.G = 1.0
    sim.add(m=1.0)                                    # star
    sim.add(m=1e-5, P=1.0, e=0.01)                    # planet 1
    sim.add(m=1e-5, P=1.3, e=0.005)                   # planet 2
    sim.add(m=1e-5, P=1.7, e=0.01)                    # planet 3
    sim.move_to_com()

    res = amd_crit_from_sim(sim, with_inclination=False, verbose=False)
    print("c        =", res["c"])
    print("AMD       =", res["AMD"])
    print("AMD_crit  =", res["AMD_crit"])
    print("beta      =", res["beta"], "->", "STABLE" if res["stable"] else "UNSTABLE")

except ImportError:
    print("rebound not installed - skipping section B.")

rebound not installed - skipping section B.


## C. Error handling

When the configuration cannot saturate `tau = 1` (e.g. very wide spacing or
large masses where the 3-body term alone exceeds 1), the optimizer raises a
clear `AMDOptimizationError` instead of returning a silently wrong result.

In [5]:
try:
    optimize_amd_crit(periods=[1.0, 1.7, 2.9], masses=[3e-6]*3)
except AMDOptimizationError as e:
    print("Caught expected error:")
    print(" ", e)

## D. Method note: trust-constr vs SLSQP

For this problem the equality constraint $\tau=1$ has a singularity at
$c_i+c_{i+1}=\delta_{i,i+1}$. **`trust-constr`** (the default) handles it
robustly; **`SLSQP`** is faster in general but tends to stall here. You can
still force it: `optimize_amd_crit(..., method='SLSQP')`.